<a href="https://colab.research.google.com/github/Aeman-Imtiaz/Parallel-and-Distributed-Computing/blob/main/MPI_Program_(2_Processes).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MPI Program (2 Processes)**

In [1]:
%%bash

cat > mpi_sum.c <<'EOF'
#include <stdio.h>
#include <mpi.h>

int main(int argc, char *argv[])
{
    int rank, size;

    MPI_Init(&argc, &argv);

    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if(size != 2)
    {
        if(rank == 0)
            printf("Please run with exactly 2 processes.\n");

        MPI_Finalize();
        return 0;
    }

    if(rank == 0)
    {
        int arr[5] = {10, 20, 30, 40, 50};

        printf("Process 0: Array = {10,20,30,40,50}\n");

        int partial_sum0 = arr[0] + arr[1];

        printf("Process 0: Computing first half -> 10 + 20 = %d\n", partial_sum0);

        printf("Process 0: Sending second half {30,40,50} to Process 1\n");

        MPI_Send(&arr[2], 3, MPI_INT, 1, 0, MPI_COMM_WORLD);

        int partial_sum1;

        MPI_Recv(&partial_sum1, 1, MPI_INT, 1, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        printf("Process 0: Received partial sum %d from Process 1\n", partial_sum1);

        int total = partial_sum0 + partial_sum1;

        printf("\nFinal Total Sum = %d\n", total);
    }
    else
    {
        int data[3];

        MPI_Recv(data, 3, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        printf("Process 1: Received {30,40,50} from Process 0\n");

        int partial_sum1 = data[0] + data[1] + data[2];

        printf("Process 1: Computing sum -> 30 + 40 + 50 = %d\n", partial_sum1);

        printf("Process 1: Sending partial sum %d back to Process 0\n", partial_sum1);

        MPI_Send(&partial_sum1, 1, MPI_INT, 0, 1, MPI_COMM_WORLD);
    }

    MPI_Finalize();
    return 0;
}
EOF

mpicc mpi_sum.c -o mpi_sum

mpirun --allow-run-as-root --oversubscribe -np 2 ./mpi_sum

Process 0: Array = {10,20,30,40,50}
Process 0: Computing first half -> 10 + 20 = 30
Process 0: Sending second half {30,40,50} to Process 1
Process 1: Received {30,40,50} from Process 0
Process 1: Computing sum -> 30 + 40 + 50 = 120
Process 1: Sending partial sum 120 back to Process 0
Process 0: Received partial sum 120 from Process 1

Final Total Sum = 150
